In [ ]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

import matplotlib.pyplot as plt

print("Libraries imported successfully")

In [ ]:
CSV_PATH = "dataset/raw_wholesale_customers.csv"
OUT_PATH = "dataset/segmented_wholesale_customers.csv"

df = pd.read_csv(CSV_PATH)

print("=== DATASET HEAD ===")
display(df.head())

print("\n=== DATASET INFO ===")
df.info()

In [ ]:
FEATURES = [
    "Fresh",
    "Milk",
    "Grocery",
    "Frozen",
    "Detergents_Paper",
    "Delicassen"
]


X = df[FEATURES].copy()


print("Selected clustering features:")
print(FEATURES)

print("\nFeature shape:")
print(X.shape)

In [ ]:
def iqr_cap(series, k=1.5):

    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)

    iqr = q3-q1

    lower = q1 - k*iqr
    upper = q3 + k*iqr

    return series.clip(lower, upper)


for col in FEATURES:
    X[col] = iqr_cap(X[col])


df[FEATURES] = X


print("IQR capping completed")
print(X.head())

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


print("Scaling completed")
print("Scaled shape:", X_scaled.shape)

In [ ]:
sse = []

print("=== ELBOW METHOD ===")


for k in range(1,11):

    model = KMeans(
        n_clusters=k,
        n_init="auto",
        random_state=42
    )

    model.fit(X_scaled)

    sse.append(model.inertia_)

    print(
        f"k={k} SSE={model.inertia_:.2f}"
    )

In [ ]:
plt.figure(figsize=(7,4))

plt.plot(
    range(1,11),
    sse,
    marker="o"
)

plt.xlabel("Number of clusters")
plt.ylabel("SSE")
plt.title("Elbow Method")

plt.show()

In [ ]:
kmeans = KMeans(
    n_clusters=5,
    n_init="auto",
    random_state=42
)


kmeans_labels = kmeans.fit_predict(X_scaled)


df["KMeans_Cluster"] = kmeans_labels


print("K-Means training completed")

display(
    df.head()
)

In [ ]:
kmeans_silhouette = silhouette_score(
    X_scaled,
    kmeans_labels
)


kmeans_dbi = davies_bouldin_score(
    X_scaled,
    kmeans_labels
)


print("=== K-MEANS METRICS ===")

print(
    "Silhouette Score:",
    round(kmeans_silhouette,3)
)

print(
    "Davies-Bouldin Index:",
    round(kmeans_dbi,3)
)

In [ ]:
centers_scaled = kmeans.cluster_centers_


centers_original = scaler.inverse_transform(
    centers_scaled
)


centers_df = pd.DataFrame(
    centers_original,
    columns=FEATURES
)


centers_df.index.name="Cluster"


print("=== CLUSTER CENTERS ===")

display(
    centers_df.round(2)
)

## Second Algorithm: Agglomerative Clustering

I chose Agglomerative (Hierarchical) Clustering as my second clustering algorithm.

### Why this algorithm fits wholesale customer segmentation:

Agglomerative Clustering creates groups by gradually merging customers that are similar based on their spending behavior. It is useful for customer segmentation because it can reveal natural groups of customers without requiring the assumption that clusters are perfectly round like K-Means.

### Research Source:

YouTube tutorial:

"Hierarchical Clustering Explained - Agglomerative Clustering"

Source:
https://www.youtube.com/watch?v=7xHsRkOdVwo

The video helped me understand how hierarchical clustering works, how clusters are merged, and when it can be used instead of K-Means.

In [ ]:
agg = AgglomerativeClustering(
    n_clusters=5
)


agg_labels = agg.fit_predict(
    X_scaled
)


df["Agglomerative_Cluster"] = agg_labels


print("Agglomerative clustering completed")

In [ ]:
agg_silhouette = silhouette_score(
    X_scaled,
    agg_labels
)


print("=== MODEL COMPARISON ===")


print(
    "K-Means Silhouette:",
    round(kmeans_silhouette,3)
)


print(
    "Agglomerative Silhouette:",
    round(agg_silhouette,3)
)

In [ ]:
sample = df.loc[
    [0,1,2],
    [
        "Channel",
        "Region",
        "Fresh",
        "Milk",
        "Grocery",
        "Frozen",
        "Detergents_Paper",
        "Delicassen",
        "KMeans_Cluster",
        "Agglomerative_Cluster"
    ]
]


display(sample)

In [ ]:
df.to_csv(
    OUT_PATH,
    index=False
)


print(
    "Saved file:",
    OUT_PATH
)